In [0]:
%sql
select * from gizmobox.bronze.v_orders

In [0]:
create or replace temporary view order_fixed as
select regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "\$1"') 
AS fixed_value 
from gizmobox.bronze.v_orders

### get the schema of a element in the json string column 'fixed_value' using the function schema_of_json()

In [0]:
select schema_of_json(fixed_value) from order_fixed limit 1;

### Use the result of above cell in from_json() function to create a STRUCT type JSON object

In [0]:
drop table if exists gizmobox.silver.orders_json;
create table if not exists gizmobox.silver.orders_json
as
select from_json(fixed_value, 
   'STRUCT<customer_id: INT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: INT, name: STRING, price: double, quantity: INT>>, order_date: STRING, order_id: INT, order_status: STRING, payment_method: STRING, total_amount: double, transaction_timestamp: timestamp>')
   as fixed_json_object
   from order_fixed;

### The column with JSON string is now converted into JSON object. The Data type icon beside the column name tells it is a JSON object type. And now you can expand each JSON element in this column.

In [0]:
select * from gizmobox.silver.orders_json

###Now its time to create a table structure from the JSON object.

In [0]:
select fixed_json_object.customer_id,
    fixed_json_object.order_date,
    fixed_json_object.order_id,
    fixed_json_object.order_status,
    fixed_json_object.payment_method,
    fixed_json_object.total_amount,
    fixed_json_object.transaction_timestamp,
    fixed_json_object.items
 from gizmobox.silver.orders_json

### Delete duplicate elements in the array - 'items'

In [0]:
select fixed_json_object.customer_id,
    fixed_json_object.order_date,
    fixed_json_object.order_id,
    fixed_json_object.order_status,
    fixed_json_object.payment_method,
    fixed_json_object.total_amount,
    fixed_json_object.transaction_timestamp,
    array_distinct(fixed_json_object.items) as items
 from gizmobox.silver.orders_json

###Now its time to explode the arrays inside the JSON object. You can see the total record now are 124.

In [0]:
create or replace temporary view v_orders_exploded
as
select fixed_json_object.customer_id,
    fixed_json_object.order_date,
    fixed_json_object.order_id,
    fixed_json_object.order_status,
    fixed_json_object.payment_method,
    fixed_json_object.total_amount,
    fixed_json_object.transaction_timestamp,
    explode(array_distinct(fixed_json_object.items)) as item
 from gizmobox.silver.orders_json

###Now create columns from the elements in the JSON object 'item'

In [0]:
create table if not exists gizmobox.silver.orders
as
select order_id,
   order_status,
   payment_method,
   total_amount,
   transaction_timestamp,
   customer_id,
   item.item_id,
   item.name,
   item.price,
   item.quantity,
   item.category,
   item.details.brand,
   item.details.color
from v_orders_exploded
order by order_id;